In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa

# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train"
OUT_TEST = "csv_test"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)

SR = 16000
WIN_DUR = 0.100   # 100 ms
HOP_DUR = 0.010   # 10 ms
FREQ_MAX = 2000

# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    """Return 1 if ANY wheeze overlaps window"""
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

def fill_short_zero_gaps(labels, max_gap=4):
    """
    Convert runs of 0s to 1s if they are between two 1s
    and length <= max_gap
    """
    labels = labels.copy()
    n = len(labels)
    i = 0

    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i  # first non-zero

            # Check bounded by ones
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1

    return labels

# =============================================================================
# FEATURE EXTRACTION (MATCHES FFT PLOT CODE)
# =============================================================================
def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []
    frame_labels = []

    for i, start in enumerate(range(0, len(audio) - win_len + 1, hop_len)):
        end = start + win_len
        frame = audio[start:end]

        frame_win = frame * np.hanning(len(frame))

        X = np.fft.rfft(frame_win)
        freqs = np.fft.rfftfreq(len(frame_win), d=1/SR)
        mag = np.abs(X)

        mag[0] = 0  # remove DC

        valid = freqs <= FREQ_MAX
        freqs = freqs[valid]
        mag = mag[valid]

        idx = np.argmax(mag)
        amp = mag[idx]
        freq = freqs[idx]

        t_start = start / SR
        t_end = end / SR
        label = wheeze_priority_label(labels, t_start, t_end)

        rows.append([
            Path(wav_path).name,
            t_start,
            amp,
            freq
        ])
        frame_labels.append(label)


    # ---- APPLY ZERO-GAP BIAS HERE ----
    frame_labels = np.array(frame_labels)
    frame_labels = fill_short_zero_gaps(frame_labels, max_gap=4)

    df = pd.DataFrame(rows, columns=[
        "file", "time_step_s", "amplitude", "frequency"
    ])
    df["label"] = frame_labels

    return df

# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []

for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []

for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)

# =============================================================================
# MODEL TRAINING
# =============================================================================
X_train = train_df[["amplitude", "frequency"]].values
y_train = train_df["label"].values

X_test = test_df[["amplitude", "frequency"]].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
model.fit(X_train_s, y_train)

# =============================================================================
# PREDICTION + SAVE TEST CSVs (PER FILE)
# =============================================================================
test_df["predicted_label"] = model.predict(X_test_s)
test_df["wheeze_prob"] = model.predict_proba(X_test_s)[:, 1]

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE ")
print(f"Train CSVs → {OUT_TRAIN}/")
print(f"Test CSVs  → {OUT_TEST}/")

              precision    recall  f1-score   support

      Normal       0.62      0.58      0.60      9637
      Wheeze       0.81      0.83      0.82     20183

    accuracy                           0.75     29820
   macro avg       0.71      0.71      0.71     29820
weighted avg       0.75      0.75      0.75     29820

Confusion Matrix:
 [[ 5627  4010]
 [ 3435 16748]]

DONE 
Train CSVs → csv_train/
Test CSVs  → csv_test/


In [10]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import librosa
from scipy.signal import find_peaks
from scipy.ndimage import median_filter


# =============================================================================
# CONFIGURATION
# =============================================================================
BASE_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
WAV_DIR = os.path.join(BASE_DIR, "*.wav")
TXT_DIR = os.path.join(BASE_DIR, "*_label_audacity.txt")

OUT_TRAIN = "csv_train_enhanced_15_1"
OUT_TEST = "csv_test_enhanced_15_1"
os.makedirs(OUT_TRAIN, exist_ok=True)
os.makedirs(OUT_TEST, exist_ok=True)
SMOOTH_WIN = 7   # frames (~70 ms)

SR = 16000
WIN_DUR = 0.100
HOP_DUR = 0.010
FREQ_MAX = 2000

# =============================================================================
# FILE PAIRING
# =============================================================================
wav_files = sorted(glob.glob(WAV_DIR))
txt_files = sorted(glob.glob(TXT_DIR))

wav_stems = {Path(w).stem: w for w in wav_files}
txt_stems = {Path(t).stem.replace('_label_audacity', ''): t for t in txt_files}

pairs = [(wav_stems[k], txt_stems[k]) for k in wav_stems if k in txt_stems]
train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# =============================================================================
# LABEL UTILITIES
# =============================================================================
def parse_audacity_labels(txt_path):
    labels = []
    with open(txt_path) as f:
        for line in f:
            p = line.strip().split('\t')
            if len(p) == 3:
                try:
                    s, e = float(p[0]), float(p[1])
                    lab = 1 if 'Wheeze' in p[2] else 0
                    labels.append((s, e, lab))
                except ValueError:
                    pass
    return labels

def wheeze_priority_label(labels, t_start, t_end):
    for s, e, lab in labels:
        if lab == 1 and not (e <= t_start or s >= t_end):
            return 1
    return 0

# =============================================================================
# FFT FEATURE EXTRACTION (ENHANCED)
# =============================================================================
def extract_per_file(wav_path, txt_path):
    audio, _ = librosa.load(wav_path, sr=SR)
    labels = parse_audacity_labels(txt_path)

    win_len = int(WIN_DUR * SR)
    hop_len = int(HOP_DUR * SR)

    rows = []

    for start in range(0, len(audio) - win_len + 1, hop_len):
        end = start + win_len
        frame = audio[start:end] * np.hanning(win_len)

        X = np.fft.rfft(frame)
        freqs = np.fft.rfftfreq(win_len, d=1/SR)
        mag = np.abs(X)
        mag[0] = 0

        valid = freqs <= FREQ_MAX
        freqs, mag = freqs[valid], mag[valid]

        # ---- Core FFT features ----
        # ---- Total spectral energy ----
        total_energy = np.sum(mag ** 2) + 1e-12

        # ---- Normalized peak amplitude ----
        idx = np.argmax(mag)
        norm_amp = mag[idx] / total_energy

        # ---- Wheeze band energy (100 Hz – 1 kHz) ----
        wheeze_band = (freqs >= 100) & (freqs <= 1000)
        band_energy = np.sum((mag[wheeze_band]) ** 2)

        band_energy_ratio = band_energy / total_energy



        # ---- Spectral entropy ----
        p = mag / (np.sum(mag) + 1e-12)
        spec_entropy = -np.sum(p * np.log(p + 1e-12))

        # ---- Spectral bandwidth ----
        centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-12)
        spec_bandwidth = np.sqrt(
            np.sum(((freqs - centroid) ** 2) * mag) / (np.sum(mag) + 1e-12)
        )

        # ---- Peak dominance ----
        peak_to_mean = mag[idx]/ (np.mean(mag) + 1e-12)

        # ---- Number of significant peaks ----
        peaks, _ = find_peaks(mag, height=0.3 * mag[idx])
        n_peaks = len(peaks)

        t_start = start / SR
        t_end = end / SR
        label = wheeze_priority_label(labels, t_start, t_end)

        rows.append([
            Path(wav_path).name,
            t_start,
            norm_amp,
            band_energy_ratio,
            spec_entropy,
            spec_bandwidth,
            peak_to_mean,
            n_peaks,
            label
        ])

    return pd.DataFrame(rows, columns=[
         "file", "time_step_s",
        "norm_amplitude", "band_energy_ratio",
        "spec_entropy", "spec_bandwidth",
        "peak_to_mean", "n_peaks",
        "label"
    ])

# =============================================================================
# PROCESS TRAIN FILES
# =============================================================================
train_frames = []
for wav, txt in train_pairs:
    df = extract_per_file(wav, txt)
    df.to_csv(os.path.join(OUT_TRAIN, Path(wav).stem + "_train.csv"), index=False)
    train_frames.append(df)

train_df = pd.concat(train_frames, ignore_index=True)

# =============================================================================
# PROCESS TEST FILES
# =============================================================================
test_frames = []
for wav, txt in test_pairs:
    df = extract_per_file(wav, txt)
    test_frames.append(df)

test_df = pd.concat(test_frames, ignore_index=True)

# =============================================================================
# MODEL TRAINING
# =============================================================================
feature_cols = [
    "norm_amplitude", "band_energy_ratio",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean", "n_peaks"
]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_test = test_df[feature_cols].values
y_test = test_df["label"].values
# Flip labels: Normal=1, Wheeze=0
y_train_flipped = 1 - y_train
y_test_flipped = 1 - y_test

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

n_pos = np.sum(y_train_flipped == 1)   # Normal frames
n_neg = np.sum(y_train_flipped == 0)   # Wheeze frames

scale_pos_weight = n_neg / n_pos
print(f"scale_pos_weight (Normal) = {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train_s, y_train_flipped)

# =============================================================================
# PREDICTION + TEMPORAL SMOOTHING
# =============================================================================
# Probabilities correspond to flipped "Normal"
normal_prob = model.predict_proba(X_test_s)[:, 1]

# Convert back to wheeze probability
test_df["wheeze_prob"] = 1 - normal_prob

# ---- Temporal smoothing per file ----
test_df["wheeze_prob_smooth"] = (
    test_df.groupby("file")["wheeze_prob"]
    .transform(lambda x: median_filter(x, size=SMOOTH_WIN))
)

# ---- Final decision ----
test_df["predicted_label_raw"] = (
    test_df["wheeze_prob"] > 0.35
).astype(int)


def enforce_min_wheeze_duration(labels, min_len=10):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels


# ---- Apply temporal constraints per file ----
final_preds = []

for fname, df_f in test_df.groupby("file"):
    preds = df_f["predicted_label_raw"].values
    # peaks = df_f["n_peaks"].values
    # amp= df_f["amplitude"].values

    # preds = enforce_min_wheeze_duration(preds, min_len=4)
    preds = enforce_min_wheeze_duration(preds, min_len=4)

    # Fill short zero gaps (<= 40 ms)
    preds = fill_short_zero_gaps(preds, max_gap=4)
    
    # Minimum wheeze duration (>= 100 ms)
    preds = enforce_min_wheeze_duration(preds, min_len=15)

    # NEW BIAS: suppress segments without low-peak frames
    # preds = suppress_segments_without_low_peaks(preds,peaks,amp,peak_thresh=4,amp_thresh=5)


    # Re-enforce duration (robustness)
    # preds = enforce_min_wheeze_duration(preds, min_len=10)

    final_preds.append(pd.Series(preds, index=df_f.index))

test_df["predicted_label"] = pd.concat(final_preds).sort_index()

for fname, df_f in test_df.groupby("file"):
    out = os.path.join(OUT_TEST, Path(fname).stem + "_test.csv")
    df_f.to_csv(out, index=False)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(y_test, test_df["predicted_label"],
                            target_names=["Normal", "Wheeze"]))
print("Confusion Matrix:\n",
      confusion_matrix(y_test, test_df["predicted_label"]))

print("\nDONE")


scale_pos_weight (Normal) = 1.44
              precision    recall  f1-score   support

      Normal       0.63      0.67      0.65      9298
      Wheeze       0.84      0.82      0.83     20522

    accuracy                           0.78     29820
   macro avg       0.74      0.75      0.74     29820
weighted avg       0.78      0.78      0.78     29820

Confusion Matrix:
 [[ 6192  3106]
 [ 3592 16930]]

DONE


In [11]:
import torch
from torch.utils.data import Dataset
import numpy as np

class WheezeSequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len=50):
        self.seq_len = seq_len
        self.feature_cols = feature_cols
        self.samples = []

        for _, g in df.groupby("file"):
            X = g[feature_cols].values
            y = g["label"].values

            for i in range(len(g) - seq_len):
                x_seq = X[i:i+seq_len]
                y_seq = y[i + seq_len // 2]  # center frame
                self.samples.append((x_seq, y_seq))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )


In [12]:
import torch.nn as nn

class WheezeCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),

            nn.Linear(128, 1)
        )

    def forward(self, x):
        # x: (batch, time, features)
        x = x.transpose(1, 2)  # → (batch, features, time)
        return self.net(x).squeeze(1)


In [13]:
from torch.utils.data import DataLoader

SEQ_LEN = 50
BATCH_SIZE = 256

feature_cols = [
    "norm_amplitude", "band_energy_ratio",
    "spec_entropy", "spec_bandwidth",
    "peak_to_mean", "n_peaks"
]

train_dataset = WheezeSequenceDataset(
    train_df, feature_cols, seq_len=SEQ_LEN
)

test_dataset = WheezeSequenceDataset(
    test_df, feature_cols, seq_len=SEQ_LEN
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)


In [14]:
import torch

y_train_seq = np.array([y for _, y in train_dataset.samples])

n_pos = np.sum(y_train_seq == 1)
n_neg = np.sum(y_train_seq == 0)

pos_weight = torch.tensor(n_neg / n_pos)
print("pos_weight:", pos_weight.item())

pos_weight: 0.6785334672898558


In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = WheezeCNN(n_features=len(feature_cols)).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/20, Loss: 0.4364
Epoch 2/20, Loss: 0.4031
Epoch 3/20, Loss: 0.3917
Epoch 4/20, Loss: 0.3798
Epoch 5/20, Loss: 0.3701
Epoch 6/20, Loss: 0.3592
Epoch 7/20, Loss: 0.3502
Epoch 8/20, Loss: 0.3456
Epoch 9/20, Loss: 0.3374
Epoch 10/20, Loss: 0.3339
Epoch 11/20, Loss: 0.3297
Epoch 12/20, Loss: 0.3262
Epoch 13/20, Loss: 0.3206
Epoch 14/20, Loss: 0.3159
Epoch 15/20, Loss: 0.3101
Epoch 16/20, Loss: 0.3077
Epoch 17/20, Loss: 0.3037
Epoch 18/20, Loss: 0.2974
Epoch 19/20, Loss: 0.2948
Epoch 20/20, Loss: 0.2889


In [35]:
model.eval()
all_probs = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = model(X)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.numpy())

y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

THRESH = 0.1  # tune 0.5–0.7
y_pred = (y_prob > THRESH).astype(int)

In [36]:
from sklearn.metrics import classification_report, confusion_matrix


print(classification_report(
    y_true, y_pred, target_names=["Normal", "Wheeze"]
))

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

      Normal       0.72      0.63      0.67      8661
      Wheeze       0.85      0.89      0.87     20159

    accuracy                           0.81     28820
   macro avg       0.78      0.76      0.77     28820
weighted avg       0.81      0.81      0.81     28820

Confusion Matrix:
 [[ 5439  3222]
 [ 2146 18013]]


In [44]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

# =============================================================================
# TEMPORAL EXTENT BIAS FUNCTIONS
# =============================================================================
def enforce_min_wheeze_duration(labels, min_len=10):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 1:
            start = i
            while i < n and labels[i] == 1:
                i += 1
            end = i
            if (end - start) < min_len:
                labels[start:end] = 0
        else:
            i += 1
    return labels


def fill_short_zero_gaps(labels, max_gap=4):
    labels = labels.copy()
    n = len(labels)
    i = 0
    while i < n:
        if labels[i] == 0:
            start = i
            while i < n and labels[i] == 0:
                i += 1
            end = i
            if start > 0 and end < n:
                if labels[start - 1] == 1 and labels[end] == 1:
                    if (end - start) <= max_gap:
                        labels[start:end] = 1
        else:
            i += 1
    return labels


# =============================================================================
# MODEL INFERENCE
# =============================================================================
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        logits = model(X)
        probs = torch.sigmoid(logits).view(-1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(y.numpy())

y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

# =============================================================================
# THRESHOLDING
# =============================================================================
THRESH = 0.1
y_pred_raw = (y_prob > THRESH).astype(int)

# =============================================================================
# TEMPORAL EXTENT BIAS (GLOBAL SEQUENCE)
# =============================================================================
# Assumes sequential frames (no shuffle)
y_pred = enforce_min_wheeze_duration(y_pred_raw, min_len=3)
y_pred = fill_short_zero_gaps(y_pred, max_gap=3)
y_pred = enforce_min_wheeze_duration(y_pred, min_len=15)

# =============================================================================
# METRICS
# =============================================================================
print(classification_report(
    y_true,
    y_pred,
    target_names=["Normal", "Wheeze"]
))

print("Confusion Matrix:\n",
      confusion_matrix(y_true, y_pred))


              precision    recall  f1-score   support

      Normal       0.71      0.63      0.67      8661
      Wheeze       0.85      0.89      0.87     20159

    accuracy                           0.81     28820
   macro avg       0.78      0.76      0.77     28820
weighted avg       0.81      0.81      0.81     28820

Confusion Matrix:
 [[ 5454  3207]
 [ 2210 17949]]


In [34]:
model.eval()

train_probs = []
train_labels = []

with torch.no_grad():
    for X, y in train_loader:
        X = X.to(device)
        logits = model(X)
        probs = torch.sigmoid(logits)

        train_probs.append(probs.cpu().numpy())
        train_labels.append(y.numpy())

y_train_prob = np.concatenate(train_probs)
y_train_true = np.concatenate(train_labels)

THRESH = 0.2  # same as test
y_train_pred = (y_train_prob > THRESH).astype(int)

print("CNN TRAIN METRICS")
print(classification_report(
    y_train_true,
    y_train_pred,
    target_names=["Normal", "Wheeze"]
))

print("Confusion Matrix:\n",
      confusion_matrix(y_train_true, y_train_pred))


CNN TRAIN METRICS
              precision    recall  f1-score   support

      Normal       0.75      0.75      0.75     46601
      Wheeze       0.83      0.83      0.83     68679

    accuracy                           0.80    115280
   macro avg       0.79      0.79      0.79    115280
weighted avg       0.80      0.80      0.80    115280

Confusion Matrix:
 [[35072 11529]
 [11822 56857]]
